# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
# Notebook-local constants and metadata
TASK_ID = 'task041'
MODEL_VERSION = 'task041-diagonal-to-filled-band-completion'
FAMILY = 'arc_static_feature_tree'
SUBTYPE = 'diagonal-to-filled-band completion'


In [2]:
# ONNX dependency setup.
import importlib.util, subprocess, sys
required = {'onnx':'onnx', 'onnxruntime':'onnxruntime', 'sklearn':'scikit-learn', 'torch':'torch'}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import onnx, onnxruntime as ort
print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.2 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [3]:
# Shared notebook builder code for ARC NeuroGolf static ONNX feature-tree models.
import json, os, zipfile, math, subprocess, sys
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.tree import DecisionTreeClassifier
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto, numpy_helper

H = W = 30
CH = 10
FORBIDDEN = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
torch.set_num_threads(2)


def find_task_json(task_id):
    candidates = [
        Path.cwd() / f'{task_id}.json',
        Path('/mnt/data') / f'{task_id}.json',
        Path('/kaggle/input') / f'{task_id}.json',
        Path('/kaggle/working') / f'{task_id}.json',
    ]
    for p in candidates:
        if p.exists():
            return p
    # recursive Kaggle fallback
    for base in [Path('/kaggle/input'), Path.cwd()]:
        if base.exists():
            hits = list(base.rglob(f'{task_id}.json'))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'Could not locate {task_id}.json')


def load_task(task_id):
    path = find_task_json(task_id)
    with open(path) as f:
        return json.load(f), path


def examples_for_scope(task, scope='all'):
    if scope == 'visible':
        return task.get('train', []) + task.get('test', [])
    if scope == 'train':
        return task.get('train', [])
    return task.get('train', []) + task.get('test', []) + task.get('arc-gen', [])


def grid_to_tensor(grid):
    arr = np.zeros((1, CH, H, W), np.float32)
    for r, row in enumerate(grid[:H]):
        for c, v in enumerate(row[:W]):
            arr[0, int(v), r, c] = 1.0
    return arr


def expected_tensor(grid):
    return grid_to_tensor(grid)


def pad_shift(t, dr, dc):
    out = torch.zeros_like(t)
    r0 = max(0, -dr); r1 = min(H, H-dr)
    c0 = max(0, -dc); c1 = min(W, W-dc)
    if r1 > r0 and c1 > c0:
        out[:, :, r0+dr:r1+dr, c0+dc:c1+dc] = t[:, :, r0:r1, c0:c1]
    return out


class LocalFeatureExtractor(nn.Module):
    def __init__(self, radius=4):
        super().__init__()
        self.radius = radius
        rr = torch.arange(H, dtype=torch.float32).view(1, 1, H, 1).expand(1, 1, H, W)
        cc = torch.arange(W, dtype=torch.float32).view(1, 1, 1, W).expand(1, 1, H, W)
        self.register_buffer('rr', rr)
        self.register_buffer('cc', cc)
        self.register_buffer('colors', torch.arange(CH, dtype=torch.float32).view(1, CH, 1, 1))

    def forward(self, x):
        B = x.shape[0]
        active = (x.sum(1, keepdim=True) > 0).float()
        color = (x * self.colors).sum(1, keepdim=True)
        row_has = (active.sum(3, keepdim=True) > 0).float()
        col_has = (active.sum(2, keepdim=True) > 0).float()
        h = row_has.sum((2, 3), keepdim=True).expand(B, 1, H, W)
        w = col_has.sum((2, 3), keepdim=True).expand(B, 1, H, W)
        rr = self.rr.expand(B, 1, H, W)
        cc = self.cc.expand(B, 1, H, W)
        feats = [rr/29.0, cc/29.0, h/30.0, w/30.0, color/9.0, active, x]
        feats += [x.sum(3, keepdim=True).expand(B, CH, H, W)/30.0,
                  x.sum(2, keepdim=True).expand(B, CH, H, W)/30.0]
        for dr in range(-self.radius, self.radius+1):
            for dc in range(-self.radius, self.radius+1):
                if dr == 0 and dc == 0:
                    continue
                if abs(dr) + abs(dc) <= self.radius:
                    feats.append(pad_shift(color/9.0, dr, dc))
                    feats.append(pad_shift(active, dr, dc))
        feat = torch.cat(feats, 1)
        return feat.permute(0, 2, 3, 1).reshape(B*H*W, -1), active


class GenericFeatureExtractor(nn.Module):
    def __init__(self, radius=4):
        super().__init__()
        self.radius = radius
        rr = torch.arange(H, dtype=torch.float32).view(1, 1, H, 1).expand(1, 1, H, W)
        cc = torch.arange(W, dtype=torch.float32).view(1, 1, 1, W).expand(1, 1, H, W)
        self.register_buffer('rr', rr)
        self.register_buffer('cc', cc)
        self.register_buffer('colors', torch.arange(CH, dtype=torch.float32).view(1, CH, 1, 1))

    def forward(self, x):
        B = x.shape[0]
        active = (x.sum(1, keepdim=True) > 0).float()
        color = (x * self.colors).sum(1, keepdim=True)
        row_has = (active.sum(3, keepdim=True) > 0).float()
        col_has = (active.sum(2, keepdim=True) > 0).float()
        h = row_has.sum((2, 3), keepdim=True).expand(B, 1, H, W)
        w = col_has.sum((2, 3), keepdim=True).expand(B, 1, H, W)
        rr = self.rr.expand(B, 1, H, W)
        cc = self.cc.expand(B, 1, H, W)
        feats = [rr/29.0, cc/29.0, h/30.0, w/30.0, color/9.0, active, x]
        feats += [x.sum(3, keepdim=True).expand(B, CH, H, W)/30.0,
                  x.sum(2, keepdim=True).expand(B, CH, H, W)/30.0,
                  x.sum((2, 3), keepdim=True).expand(B, CH, H, W)/900.0]
        feats += [torch.cumsum(x, dim=3)/30.0,
                  (x.sum(3, keepdim=True)-torch.cumsum(x, dim=3))/30.0,
                  torch.cumsum(x, dim=2)/30.0,
                  (x.sum(2, keepdim=True)-torch.cumsum(x, dim=2))/30.0]
        for dr in range(-self.radius, self.radius+1):
            for dc in range(-self.radius, self.radius+1):
                if dr == 0 and dc == 0:
                    continue
                if abs(dr) + abs(dc) <= self.radius or (abs(dr) == abs(dc) and abs(dr) <= self.radius):
                    feats.append(pad_shift(color/9.0, dr, dc))
                    feats.append(pad_shift(active, dr, dc))
        feats += [torch.flip(color/9.0, dims=[2]), torch.flip(color/9.0, dims=[3]), torch.flip(color/9.0, dims=[2, 3])]
        for m in [2,3,4,5,6,7,8,9,10]:
            feats.append((rr % m)/float(m))
            feats.append((cc % m)/float(m))
            feats.append(((rr+cc) % m)/float(m))
            feats.append(((rr-cc+30) % m)/float(m))
        feat = torch.cat(feats, 1)
        return feat.permute(0, 2, 3, 1).reshape(B*H*W, -1), active


class PeriodicFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        rr = torch.arange(H, dtype=torch.float32).view(1, 1, H, 1).expand(1, 1, H, W)
        cc = torch.arange(W, dtype=torch.float32).view(1, 1, 1, W).expand(1, 1, H, W)
        self.register_buffer('rr', rr)
        self.register_buffer('cc', cc)
        self.register_buffer('colors', torch.arange(CH, dtype=torch.float32).view(1, CH, 1, 1))

    def forward(self, x):
        B = x.shape[0]
        active = (x.sum(1, keepdim=True) > 0).float()
        color = (x * self.colors).sum(1, keepdim=True)
        rr = self.rr.expand(B, 1, H, W)
        cc = self.cc.expand(B, 1, H, W)
        row_has = (active.sum(3, keepdim=True) > 0).float()
        col_has = (active.sum(2, keepdim=True) > 0).float()
        h = row_has.sum((2, 3), keepdim=True).expand(B, 1, H, W)
        w = col_has.sum((2, 3), keepdim=True).expand(B, 1, H, W)
        feats = [rr/29.0, cc/29.0, h/30.0, w/30.0, color/9.0, active, x]
        feats += [x.sum(3, keepdim=True).expand(B, CH, H, W)/30.0,
                  x.sum(2, keepdim=True).expand(B, CH, H, W)/30.0,
                  x.sum((2, 3), keepdim=True).expand(B, CH, H, W)/900.0]
        for m in range(2, 13):
            feats += [(rr % m)/m, (cc % m)/m, ((rr+cc) % m)/m, ((rr-cc+60) % m)/m]
        feat = torch.cat(feats, 1)
        return feat.permute(0, 2, 3, 1).reshape(B*H*W, -1), active


class RowIntervalFill(nn.Module):
    """Horizontal same-colour interval closure.

    For each non-background colour independently, a cell becomes that colour
    when the same row contains that colour on both its left and right side
    including the cell itself. This fills the interior of diagonal/chevron
    border rows without memorising any visible output grid.
    """
    def forward(self, x):
        active = (x.sum(1, keepdim=True) > 0.0).float()
        filled_channels = []
        for k in range(1, CH):
            m = x[:, k:k+1]
            seen_left = (torch.cumsum(m, dim=3) > 0.0).float()
            seen_right = torch.flip((torch.cumsum(torch.flip(m, dims=[3]), dim=3) > 0.0).float(), dims=[3])
            filled_channels.append(seen_left * seen_right * active)
        colored = torch.cat(filled_channels, dim=1)
        occupied = (colored.sum(1, keepdim=True) > 0.0).float()
        background = (1.0 - occupied) * active
        return torch.cat([background, colored], dim=1)


class Task112Reflect(nn.Module):
    def __init__(self):
        super().__init__()
        rr = torch.arange(H, dtype=torch.float32).view(1, H, 1).expand(1, H, W)
        cc = torch.arange(W, dtype=torch.float32).view(1, 1, W).expand(1, H, W)
        self.register_buffer('rr', rr)
        self.register_buffer('cc', cc)
        self.register_buffer('colors', torch.arange(CH, dtype=torch.float32).view(1, CH, 1, 1))

    def sample_mask(self, m, rsrc, csrc):
        gx = 2.0*csrc/(W-1)-1.0
        gy = 2.0*rsrc/(H-1)-1.0
        grid = torch.stack([gx, gy], dim=-1)
        return F.grid_sample(m, grid, mode='nearest', padding_mode='zeros', align_corners=True)

    def forward(self, x):
        B = x.shape[0]
        active = (x.sum(1, keepdim=True) > 0).float()
        m2 = x[:, 2:3]
        sep = x[:, 3:4]
        rows = self.rr.expand(B, H, W)
        cols = self.cc.expand(B, H, W)
        sep2 = sep[:, 0]
        big = torch.full_like(rows, 100.0)
        small = torch.full_like(rows, -100.0)
        rmin = torch.amin(torch.where(sep2 > 0.5, rows, big).reshape(B, -1), dim=1).view(B, 1, 1)
        rmax = torch.amax(torch.where(sep2 > 0.5, rows, small).reshape(B, -1), dim=1).view(B, 1, 1)
        cmin = torch.amin(torch.where(sep2 > 0.5, cols, big).reshape(B, -1), dim=1).view(B, 1, 1)
        cmax = torch.amax(torch.where(sep2 > 0.5, cols, small).reshape(B, -1), dim=1).view(B, 1, 1)
        rsum = rmin + rmax
        csum = cmin + cmax
        v = self.sample_mask(m2, rows, csum-cols)
        h = self.sample_mask(m2, rsum-rows, cols)
        hv = self.sample_mask(m2, rsum-rows, csum-cols)
        out2 = ((m2+v+h+hv) > 0.5).float()*(1-sep)
        outs = []
        for k in range(CH):
            if k == 2:
                outs.append(out2)
            elif k == 3:
                outs.append(sep)
            else:
                outs.append(x[:, k:k+1]*(1-out2)*(1-sep))
        return torch.cat(outs, 1)*active


def make_tree_ensemble_classifier_node(clf):
    tree = clf.tree_
    classes = [int(c) for c in clf.classes_]
    class_to_index = {label: idx for idx, label in enumerate(classes)}
    nodes_treeids=[]; nodes_nodeids=[]; nodes_featureids=[]; nodes_modes=[]; nodes_values=[]
    nodes_truenodeids=[]; nodes_falsenodeids=[]; nodes_missing_value_tracks_true=[]; nodes_hitrates=[]
    class_treeids=[]; class_nodeids=[]; class_ids=[]; class_weights=[]
    for node_id in range(tree.node_count):
        left = int(tree.children_left[node_id]); right = int(tree.children_right[node_id])
        is_leaf = left == right or left < 0
        nodes_treeids.append(0); nodes_nodeids.append(node_id)
        nodes_featureids.append(0 if is_leaf else int(tree.feature[node_id]))
        nodes_modes.append('LEAF' if is_leaf else 'BRANCH_LEQ')
        nodes_values.append(0.0 if is_leaf else float(tree.threshold[node_id]))
        nodes_truenodeids.append(0 if is_leaf else left)
        nodes_falsenodeids.append(0 if is_leaf else right)
        nodes_missing_value_tracks_true.append(0); nodes_hitrates.append(1.0)
        if is_leaf:
            counts = tree.value[node_id][0]
            total = float(np.sum(counts)) or 1.0
            for class_pos, label in enumerate(classes):
                class_treeids.append(0); class_nodeids.append(node_id)
                class_ids.append(class_to_index[label])
                class_weights.append(float(counts[class_pos]) / total)
    return helper.make_node(
        'TreeEnsembleClassifier', ['features'], ['label', 'probabilities'], domain='ai.onnx.ml',
        classlabels_int64s=classes,
        nodes_treeids=nodes_treeids, nodes_nodeids=nodes_nodeids, nodes_featureids=nodes_featureids,
        nodes_modes=nodes_modes, nodes_values=nodes_values, nodes_truenodeids=nodes_truenodeids,
        nodes_falsenodeids=nodes_falsenodeids, nodes_missing_value_tracks_true=nodes_missing_value_tracks_true,
        nodes_hitrates=nodes_hitrates, class_treeids=class_treeids, class_nodeids=class_nodeids,
        class_ids=class_ids, class_weights=class_weights, post_transform='NONE')


def make_dataset(task, extractor, scope):
    examples = examples_for_scope(task, scope)
    inputs=[]; labels=[]
    for ex in examples:
        inputs.append(grid_to_tensor(ex['input'])[0])
        lab = np.zeros(H*W, dtype=np.int64)
        for r, row in enumerate(ex['output'][:H]):
            for c, v in enumerate(row[:W]):
                lab[r*W+c] = int(v)
        labels.append(lab)
    X=[]; y=[]
    with torch.no_grad():
        for st in range(0, len(inputs), 128):
            arr = np.stack(inputs[st:st+128], 0)
            feats, active = extractor(torch.from_numpy(arr))
            feats = feats.numpy().reshape(len(arr), H*W, -1)
            masks = active.numpy().reshape(len(arr), H*W) > 0.5
            for i in range(len(arr)):
                X.append(feats[i][masks[i]])
                y.append(labels[st+i][masks[i]])
    return np.concatenate(X).astype(np.float32), np.concatenate(y).astype(np.int64)


def export_feature_tree(task, out_path, extractor, clf):
    tmp_features = Path(out_path).with_suffix('.features.onnx')
    dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
    torch.onnx.export(extractor.eval(), dummy, str(tmp_features), input_names=['input'], output_names=['features', 'active'],
                      opset_version=17, dynamic_axes=None, do_constant_folding=True, dynamo=False)
    feature_onnx = onnx.load(str(tmp_features))
    nodes = list(feature_onnx.graph.node)
    inits = list(feature_onnx.graph.initializer)
    nodes.append(make_tree_ensemble_classifier_node(clf))
    def init(name, array):
        inits.append(numpy_helper.from_array(np.asarray(array), name)); return name
    init('depth10', np.array(10, dtype=np.int64))
    init('onehot_values', np.array([0.0, 1.0], dtype=np.float32))
    init('shape_1_30_30_10', np.array([1, H, W, CH], dtype=np.int64))
    nodes += [
        helper.make_node('OneHot', ['label', 'depth10', 'onehot_values'], ['oh_flat'], axis=-1),
        helper.make_node('Reshape', ['oh_flat', 'shape_1_30_30_10'], ['oh_nhwc']),
        helper.make_node('Transpose', ['oh_nhwc'], ['oh_nchw'], perm=[0, 3, 1, 2]),
        helper.make_node('Mul', ['oh_nchw', 'active'], ['output']),
    ]
    graph = helper.make_graph(nodes, 'arc_static_feature_tree',
        [helper.make_tensor_value_info('input', TensorProto.FLOAT, [1, CH, H, W])],
        [helper.make_tensor_value_info('output', TensorProto.FLOAT, [1, CH, H, W])], initializer=inits)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17), helper.make_opsetid('ai.onnx.ml', 3)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    onnx.save(model, str(out_path))
    try: tmp_features.unlink()
    except Exception: pass


def export_symbolic_model(model, out_path):
    dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
    torch.onnx.export(model.eval(), dummy, str(out_path), input_names=['input'], output_names=['output'],
                      opset_version=17, dynamic_axes=None, do_constant_folding=True, dynamo=False)
    onnx_model = onnx.load(str(out_path))
    onnx_model.ir_version = 8
    onnx.checker.check_model(onnx_model)
    onnx.save(onnx_model, str(out_path))


def build_model(task, out_path, config):
    if config['builder'] == 'symbolic_row_interval_fill':
        export_symbolic_model(RowIntervalFill(), out_path)
        return {'builder': 'symbolic_row_interval_fill'}
    if config['builder'] == 'symbolic_task112':
        export_symbolic_model(Task112Reflect(), out_path)
        return {'builder': 'symbolic_task112_reflection'}
    if config['extractor'] == 'periodic':
        extractor = PeriodicFeatureExtractor()
    elif config['extractor'] == 'generic4':
        extractor = GenericFeatureExtractor(radius=4)
    elif config['extractor'] == 'local4':
        extractor = LocalFeatureExtractor(radius=4)
    elif config['extractor'] == 'local2':
        extractor = LocalFeatureExtractor(radius=2)
    else:
        raise ValueError(config['extractor'])
    X, y = make_dataset(task, extractor, config['train_scope'])
    clf = DecisionTreeClassifier(random_state=42, min_samples_leaf=1, max_depth=config['max_depth'])
    clf.fit(X, y)
    wrong = int(np.sum(clf.predict(X) != y))
    assert wrong == 0, {'wrong_training_pixels': wrong, 'nodes': int(clf.tree_.node_count)}
    export_feature_tree(task, out_path, extractor, clf)
    return {'builder': 'semantic_feature_tree', 'extractor': config['extractor'], 'train_scope': config['train_scope'],
            'tree_nodes': int(clf.tree_.node_count), 'tree_depth': int(clf.get_depth())}


def run_onnx_grid(sess, grid):
    out = sess.run(['output'], {'input': grid_to_tensor(grid)})[0]
    return (out > 0).astype(np.float32)


def validate_model(model_path, task, scope='visible'):
    sess = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
    examples = examples_for_scope(task, scope)
    right = 0; total = 0; first_wrong = None
    for i, ex in enumerate(examples):
        total += 1
        ok = np.array_equal(run_onnx_grid(sess, ex['input']), expected_tensor(ex['output']))
        if ok:
            right += 1
        elif first_wrong is None:
            first_wrong = i
    model = onnx.load(str(model_path))
    ops = {}
    for node in model.graph.node:
        ops[node.op_type] = ops.get(node.op_type, 0) + 1
    bad = sorted(FORBIDDEN & set(ops))
    return {'right': right, 'total': total, 'first_wrong': first_wrong,
            'file_size_bytes': Path(model_path).stat().st_size,
            'under_1_4mb': Path(model_path).stat().st_size < 1_400_000,
            'forbidden_ops_present': bad, 'op_counts': ops}


In [4]:
from pathlib import Path
import json

ROOT = Path.cwd()
TASK_CONFIG = {'builder': 'symbolic_row_interval_fill', 'train_scope': 'all', 'verify_scope': 'all', 'note': 'for each non-background colour, fill every horizontal gap between same-colour pixels in the same row'}
DATA_PATH = find_task_json(TASK_ID)
OUT_DIR = ROOT / f'working_submission_{TASK_ID}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT_DIR / f'{TASK_ID}.onnx'
print('DATA_PATH =', DATA_PATH)
print('OUT_DIR =', OUT_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('TASK_CONFIG =', TASK_CONFIG)

DATA_PATH = /kaggle/input/competitions/neurogolf-2026/task041.json
OUT_DIR = /kaggle/working/working_submission_task041
MODEL_PATH = /kaggle/working/working_submission_task041/task041.onnx
TASK_CONFIG = {'builder': 'symbolic_row_interval_fill', 'train_scope': 'all', 'verify_scope': 'all', 'note': 'for each non-background colour, fill every horizontal gap between same-colour pixels in the same row'}


In [5]:
task, task_path = load_task(TASK_ID)
print('task path:', task_path)
print('train examples:', len(task.get('train', [])))
print('test examples:', len(task.get('test', [])))
print('arc-gen examples:', len(task.get('arc-gen', [])))
print('first input shape:', (len(task['train'][0]['input']), len(task['train'][0]['input'][0])))
print('first output shape:', (len(task['train'][0]['output']), len(task['train'][0]['output'][0])))

task path: /kaggle/input/competitions/neurogolf-2026/task041.json
train examples: 3
test examples: 1
arc-gen examples: 262
first input shape: (10, 10)
first output shape: (10, 10)


In [6]:
# Small inspection: color counts for the first training pair.
from collections import Counter
import numpy as np
first = task['train'][0]
print('input colors:', Counter(np.array(first['input']).ravel()))
print('output colors:', Counter(np.array(first['output']).ravel()))
print('changed cells:', int(np.sum(np.array(first['input']) != np.array(first['output']))))

input colors: Counter({np.int64(0): 90, np.int64(3): 10})
output colors: Counter({np.int64(0): 78, np.int64(3): 22})
changed cells: 12


In [7]:
# Export plan for this task.
plan = {
    'task_id': TASK_ID,
    'builder': TASK_CONFIG['builder'],
    'extractor': TASK_CONFIG.get('extractor'),
    'max_depth': TASK_CONFIG.get('max_depth'),
    'train_scope': TASK_CONFIG['train_scope'],
    'verify_scope': TASK_CONFIG['verify_scope'],
    'model_path': str(MODEL_PATH),
}
plan

{'task_id': 'task041',
 'builder': 'symbolic_row_interval_fill',
 'extractor': None,
 'max_depth': None,
 'train_scope': 'all',
 'verify_scope': 'all',
 'model_path': '/kaggle/working/working_submission_task041/task041.onnx'}

In [8]:
# Task-specific model construction wrapper.
def build_current_model():
    task, _ = load_task(TASK_ID)
    return build_model(task, MODEL_PATH, TASK_CONFIG)

def validate_current_model(scope=None):
    task, _ = load_task(TASK_ID)
    return validate_model(MODEL_PATH, task, scope or TASK_CONFIG['verify_scope'])

In [9]:
# Build ONNX model and enforce competition constraints.
build_info = build_current_model()
validation_report = validate_current_model()
assert validation_report['right'] == validation_report['total'], validation_report
assert validation_report['under_1_4mb'], validation_report['file_size_bytes']
assert not validation_report['forbidden_ops_present'], validation_report['forbidden_ops_present']
print('build_info:', build_info)
print('validation_report:', validation_report)

/tmp/ipykernel_16/2580427517.py:341: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model.eval(), dummy, str(out_path), input_names=['input'], output_names=['output'],


build_info: {'builder': 'symbolic_row_interval_fill'}
validation_report: {'right': 266, 'total': 266, 'first_wrong': None, 'file_size_bytes': 21828, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Constant': 148, 'ReduceSum': 2, 'Greater': 20, 'Cast': 20, 'Slice': 27, 'CumSum': 18, 'Mul': 19, 'Concat': 2, 'Sub': 1}}


In [10]:
# Optional broader arc-gen check. This is asserted only when verify_scope == 'all'.
optional_all_report = validate_model(MODEL_PATH, task, 'all')
print('optional_all_report:', optional_all_report)
if TASK_CONFIG['verify_scope'] == 'all':
    assert optional_all_report['right'] == optional_all_report['total'], optional_all_report

optional_all_report: {'right': 266, 'total': 266, 'first_wrong': None, 'file_size_bytes': 21828, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Constant': 148, 'ReduceSum': 2, 'Greater': 20, 'Cast': 20, 'Slice': 27, 'CumSum': 18, 'Mul': 19, 'Concat': 2, 'Sub': 1}}


In [11]:
# Model/version manifest.
run_manifest = {
    'task_id': TASK_ID,
    'model_version': MODEL_VERSION,
    'strategy': TASK_CONFIG['note'],
    'build_info': build_info,
    'validation_report': validation_report,
    'optional_all_report': optional_all_report,
}
run_manifest

{'task_id': 'task041',
 'model_version': 'task041-diagonal-to-filled-band-completion',
 'strategy': 'for each non-background colour, fill every horizontal gap between same-colour pixels in the same row',
 'build_info': {'builder': 'symbolic_row_interval_fill'},
 'validation_report': {'right': 266,
  'total': 266,
  'first_wrong': None,
  'file_size_bytes': 21828,
  'under_1_4mb': True,
  'forbidden_ops_present': [],
  'op_counts': {'Constant': 148,
   'ReduceSum': 2,
   'Greater': 20,
   'Cast': 20,
   'Slice': 27,
   'CumSum': 18,
   'Mul': 19,
   'Concat': 2,
   'Sub': 1}},
 'optional_all_report': {'right': 266,
  'total': 266,
  'first_wrong': None,
  'file_size_bytes': 21828,
  'under_1_4mb': True,
  'forbidden_ops_present': [],
  'op_counts': {'Constant': 148,
   'ReduceSum': 2,
   'Greater': 20,
   'Cast': 20,
   'Slice': 27,
   'CumSum': 18,
   'Mul': 19,
   'Concat': 2,
   'Sub': 1}}}

In [12]:
# Architecture report.
model = onnx.load(str(MODEL_PATH))
op_counts = {}
for node in model.graph.node:
    op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
architecture_report = {
    'file_size_bytes': MODEL_PATH.stat().st_size,
    'nodes': len(model.graph.node),
    'op_counts': op_counts,
    'forbidden_ops_present': sorted(FORBIDDEN & set(op_counts)),
}
architecture_report

{'file_size_bytes': 21828,
 'nodes': 257,
 'op_counts': {'Constant': 148,
  'ReduceSum': 2,
  'Greater': 20,
  'Cast': 20,
  'Slice': 27,
  'CumSum': 18,
  'Mul': 19,
  'Concat': 2,
  'Sub': 1},
 'forbidden_ops_present': []}

In [13]:
# Persist metadata next to ONNX.
manifest_path = OUT_DIR / f'{TASK_ID}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote model:', MODEL_PATH)
print('wrote manifest:', manifest_path)

wrote model: /kaggle/working/working_submission_task041/task041.onnx
wrote manifest: /kaggle/working/working_submission_task041/task041_manifest.json


In [14]:
# Package single-task submission zip.
zip_path = ROOT / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, MODEL_PATH.name)
print('wrote zip:', zip_path)

wrote zip: /kaggle/working/submission.zip


In [15]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote manifest: /kaggle/working/working_submission_task041/arc_static_feature_tree_task041-diagonal-to-filled-band-completion_manifest.json
